# 05 — VizWiz Evaluation Verification

## Objective
Re-evaluate **all three existing baselines** using consistent VQA-standard answer normalization and the official 10-annotator agreement formula, without rerunning model inference.

| Model | Predictions |
|---|---|
| Constant baseline | Always `unanswerable` |
| Question-only TF-IDF + LinearSVC | Saved CSV |
| Pretrained BLIP | Saved JSONL |

**Important distinction:** The implementation in this notebook is an **independent VQA-compatible evaluator**. Compare it against the downloadable [VizWiz evaluation API](https://vizwiz.org/tasks-and-datasets/vqa/) before calling the outputs *officially verified*. The legacy EvalAI servers have been deprecated.

Our previous preliminary results (simple normalization) were: constant **52.12%**, question-only **52.55%**, BLIP **19.94%** overall.

### Evaluation steps
1. Load and match all 4,319 validation examples.
2. Apply VQA-style punctuation, article, number, and contraction normalization.
3. Recalculate accuracy using leave-one-annotator-out agreement.
4. Compare to earlier simple-normalization scores.
5. Export per-example results and a compact summary.
6. Cross-check with official VizWiz evaluation API when available.

The VizWiz *answerability challenge* uses separate confidence scores and average precision. This notebook scores **VQA answer generation**, not that second official task.

## 0 — Setup

Run in VS Code with your existing Colab kernel. GPU is not needed.

In [ ]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
from collections import Counter
from IPython.display import display

# If Google Drive is not already mounted in this Colab runtime, uncomment:
# from google.colab import drive
# drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/VQA-Project")
VAL_PATH = BASE / "data" / "Annotations" / "val.json"
QO_PATH = BASE / "results" / "question_only" / "validation_predictions.csv"
BLIP_PATH = BASE / "results" / "blip_pretrained" / "predictions_blip_vqa_base.jsonl"
OUT = BASE / "results" / "official_style_evaluation"

for p in [VAL_PATH, QO_PATH, BLIP_PATH]:
    print("FOUND" if p.is_file() else "MISSING", p)
    assert p.is_file(), f"Missing {p}"

## 1 — Load and align predictions

Always align on image filename rather than row number.

In [ ]:
with VAL_PATH.open(encoding="utf-8") as f:
    val = pd.DataFrame(json.load(f))

qo = pd.read_csv(QO_PATH)[["image", "prediction"]].rename(
    columns={"prediction": "question_only_prediction"}
)

with BLIP_PATH.open(encoding="utf-8") as f:
    blip = pd.DataFrame([
        json.loads(line) for line in f if line.strip()
    ])[["image", "prediction"]].rename(
        columns={"prediction": "blip_prediction"}
    )

assert len(val) == len(qo) == len(blip) == 4319
for name, frame in [("validation", val), ("question-only", qo), ("BLIP", blip)]:
    assert frame["image"].is_unique, f"{name}: duplicate image IDs"

expected = set(val["image"])
assert set(qo["image"]) == set(blip["image"]) == expected

df = val.merge(qo, on="image", validate="one_to_one").merge(
    blip, on="image", validate="one_to_one"
)
df["constant_prediction"] = "unanswerable"
assert len(df) == 4319
print(f"Aligned {len(df)} / 4319 examples")
display(df[["image","question","question_only_prediction","blip_prediction"]].head(3))

## 2 — VQA-compatible normalization

The canonical VQA evaluation convention normalizes punctuation, articles, numbers, and common contractions. This implementation follows those conventions and will be **checked against the official API** before publication.

Unlike our earlier `normalize_simple` function, this one may make responses such as `a dog` and `dog`, or `two` and `2`, equivalent.

The exact official normalization implementation can vary by release, so any discrepancy found during the API cross-check must be resolved rather than silently treating these results as authoritative.

In [ ]:
# Mappings and normalization rules compatible with the standard VQA evaluator.
_ARTICLES = {"a", "an", "the"}
_NUMBERS = {
    "none": "0", "zero": "0", "one": "1", "two": "2",
    "three": "3", "four": "4", "five": "5",
    "six": "6", "seven": "7", "eight": "8",
    "nine": "9", "ten": "10"
}
_CONTRACTIONS = {
    "aint": "ain't", "arent": "aren't", "cant": "can't",
    "couldnt": "couldn't", "didnt": "didn't", "doesnt": "doesn't",
    "dont": "don't", "hadnt": "hadn't", "hasnt": "hasn't",
    "havent": "haven't", "hes": "he's", "im": "i'm",
    "isnt": "isn't", "itll": "it'll", "its": "it's",
    "ive": "i've", "lets": "let's", "mightnt": "mightn't",
    "mustnt": "mustn't", "shant": "shan't", "shed": "she'd",
    "shell": "she'll", "shes": "she's", "shouldnt": "shouldn't",
    "somebodyd": "somebody'd", "somebodyll": "somebody'll",
    "somebodys": "somebody's", "thats": "that's",
    "thered": "there'd", "therere": "there're",
    "theres": "there's", "theyd": "they'd",
    "theyll": "they'll", "theyre": "they're",
    "theyve": "they've", "wasnt": "wasn't",
    "wed": "we'd", "well": "we'll", "were": "we're",
    "weve": "we've", "werent": "weren't",
    "whatll": "what'll", "whatre": "what're",
    "whats": "what's", "whatve": "what've",
    "wheres": "where's", "whod": "who'd",
    "wholl": "who'll", "whos": "who's",
    "whyd": "why'd", "whyre": "why're",
    "whys": "why's", "wont": "won't",
    "wouldnt": "wouldn't", "yall": "y'all",
    "youd": "you'd", "youll": "you'll",
    "youre": "you're", "youve": "you've"
}
_PUNCTUATION = [';', '/', '[', ']', '"', '{', '}', '(', ')',
                '=', '+', '\\', '_', '-', '>', '<', '@', '`',
                ',', '?', '!']
_COMMA_STRIP = re.compile(r'(?<=\d),(?=\d)')
_PERIOD_STRIP = re.compile(r'(?<!\d)\.|\.(?!\d)')

def normalize_vqa(value):
    text = str(value).lower().replace("\n", " ").replace("\t", " ").strip()
    text = _COMMA_STRIP.sub("", text)
    for p in _PUNCTUATION:
        if (p + " " in text) or (" " + p in text) or (p == "," and re.search(r'\d,\d', text)):
            text = text.replace(p, "")
        else:
            text = text.replace(p, " ")
    text = _PERIOD_STRIP.sub("", text)
    tokens = text.split()
    tokens = [_NUMBERS.get(t, t) for t in tokens if t not in _ARTICLES]
    tokens = [_CONTRACTIONS.get(t, t) for t in tokens]
    return " ".join(tokens)

# Spot checks; these are behavioral checks, not proof of equivalence with the API
tests = {
    "A dog": "dog",
    "two dogs": "2 dogs",
    "DON'T": "don't",
    "don't": "don't",
    "a blue shirt": "blue shirt",
    "1,000": "1000"
}
for raw, expected in tests.items():
    result = normalize_vqa(raw)
    assert result == expected, (raw, result, expected)
print("Normalization sanity checks passed.")

## 3 — Score all predictions

Each annotator is held out in turn. For the remaining nine human answers, a prediction earns `min(matches / 3, 1)`. Average the ten resulting scores.

In [ ]:
def vqa_agreement_score(prediction, references, normalizer):
    p = normalizer(prediction)
    refs = [normalizer(a["answer"]) for a in references]
    if len(refs) != 10:
        raise ValueError(f"Expected 10 answers, got {len(refs)}")
    return float(np.mean([
        min(sum(p == ref for j, ref in enumerate(refs) if j != i) / 3.0, 1.0)
        for i in range(10)
    ]))

def normalize_simple(text):
    return re.sub(r"\s+", " ", str(text).strip().lower())

for model in ("constant", "question_only", "blip"):
    prediction_col = f"{model}_prediction"
    df[f"{model}_simple"] = [
        vqa_agreement_score(pred, answers, normalize_simple)
        for pred, answers in zip(df[prediction_col], df["answers"])
    ]
    df[f"{model}_vqa_compatible"] = [
        vqa_agreement_score(pred, answers, normalize_vqa)
        for pred, answers in zip(df[prediction_col], df["answers"])
    ]

print("Scored all models using both normalization strategies.")

## 4 — Compare previous and VQA-compatible results

**Do not label these scores official until checking the released API.** Differences from the earlier numbers are expected if punctuation, articles, or number words change answer matching.

In [ ]:
rows = []
for subset_name, subset in {
    "Overall": df,
    "Answerable": df[df["answerable"].astype(int) == 1],
    "Unanswerable": df[df["answerable"].astype(int) == 0]
}.items():
    for model in ("constant", "question_only", "blip"):
        rows.append({
            "Subset": subset_name,
            "Model": model,
            "Examples": len(subset),
            "Simple (%)": round(subset[f"{model}_simple"].mean() * 100, 2),
            "VQA-compatible (%)": round(subset[f"{model}_vqa_compatible"].mean() * 100, 2),
            "Difference (pp)": round(
                (subset[f"{model}_vqa_compatible"].mean()
                 - subset[f"{model}_simple"].mean()) * 100, 2)
        })
summary = pd.DataFrame(rows)
display(summary)

print("Simple-normalization scores should reproduce your prior results:")
display(summary[summary["Subset"] == "Overall"][["Model", "Simple (%)"]])

## 5 — Inspect examples changed by normalization

This is the best way to detect normalization mistakes before writing up new results.

In [ ]:
for model in ("question_only", "blip"):
    delta = df[f"{model}_vqa_compatible"] - df[f"{model}_simple"]
    sample = df.loc[delta.abs() > 1e-9, [
        "image", "question", f"{model}_prediction",
        f"{model}_simple", f"{model}_vqa_compatible", "answers"
    ]].copy()
    print(f"\n{model}: {len(sample)} examples with changed score")
    if not sample.empty:
        sample["human_answers"] = sample["answers"].apply(
            lambda aa: Counter(a["answer"] for a in aa).most_common(5)
        )
        display(sample.drop(columns="answers").head(10))

## 6 — Save results

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT / "three_model_vqa_compatible_summary.csv", index=False)
df[[
    "image", "question", "answerable",
    "constant_prediction", "question_only_prediction", "blip_prediction",
    "constant_simple", "question_only_simple", "blip_simple",
    "constant_vqa_compatible", "question_only_vqa_compatible",
    "blip_vqa_compatible"
]].to_csv(OUT / "per_example_vqa_compatible_scores.csv", index=False)

print("Saved to:", OUT)

## 7 — Official API cross-check

The VizWiz website provides an **example evaluation API ZIP**. Its link is:

`https://vizwiz.cs.colorado.edu/VizWiz_final/vqa_data/API.zip`

The next cell downloads and *lists* the official API archive in your Colab runtime so you can identify its evaluation entry point. It does not execute unreviewed downloaded code.

Run this cell and share the printed list of API files, together with the summary from Section 4. We can then verify the exact normalization and scoring implementation against the official package and, if necessary, update the notebook.

In [ ]:
import io
import zipfile
import requests

API_URL = "https://vizwiz.cs.colorado.edu/VizWiz_final/vqa_data/API.zip"

try:
    response = requests.get(API_URL, timeout=30)
    response.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
        names = zf.namelist()
    print("Official API archive downloaded.")
    print("\n".join(names[:100]))
    print(f"Total archive entries: {len(names)}")
except Exception as exc:
    print("Could not download or inspect the official API archive:", repr(exc))
    print("You may use the API link from the VizWiz website manually.")

## 8 — Findings (complete after running)

| Model | Previous overall (%) | VQA-compatible overall (%) | Verified against official API? |
|---|---:|---:|---|
| Constant unanswerable | 52.12 | TBD | No |
| Question-only | 52.55 | TBD | No |
| Pretrained BLIP | 19.94 | TBD | No |

### Interpretation
- Did VQA-compatible normalization affect the rankings? **TBD**
- Which responses changed score because of normalization? **TBD**
- Did the released API confirm the implementation? **TBD**

### Next step
Once the scoring implementation is verified, proceed to `06_blip_finetuning.ipynb` and use **this identical evaluator** for both pretrained and fine-tuned BLIP. Keep all fine-tuning examples from the training split, not the validation split.